# Historical Hiring Analysis

Using data from the Wayback Machine scraper, which captures job postings archived over time — including roles that have since been closed or filled.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────
CSV_PATH = "anthropic_salaries_historical.csv"
# ────────────────────────────────────────────────────────────────────

import re
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

from classify import (
    classify_department, classify_seniority, classify_work_mode,
    add_classifications, add_usd_salary, TO_USD, SENIORITY_ORDER,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 6)

# Derive company name from CSV path for chart titles
COMPANY = Path(CSV_PATH).stem.replace("_salaries_historical", "").replace("_", " ").title()

## Data Loading

In [ ]:
hist = pd.read_csv(CSV_PATH, dtype={"job_id": str})
hist["first_seen"] = pd.to_datetime(hist["first_seen"], format="%Y%m%d")
hist["last_seen"] = pd.to_datetime(hist["last_seen"], format="%Y%m%d")
hist["is_active"] = hist["is_active"].astype(bool)

add_classifications(hist)
add_usd_salary(hist)

hist_salary = hist.dropna(subset=["salary_min", "salary_max"]).copy()

seniority_order = SENIORITY_ORDER

print(f"Historical dataset: {len(hist):,} total jobs")
print(f"  With salary data: {len(hist_salary):,}")
print(f"  Currently active: {hist['is_active'].sum():,}")
print(f"  Closed / filled:  {(~hist['is_active']).sum():,}")
print(f"  Date range: {hist['first_seen'].min():%Y-%m-%d} to {hist['last_seen'].max():%Y-%m-%d}")
hist.head()

## 10. Hiring Volume Over Time

How many new job postings appeared each month? This proxies for hiring intensity.

In [ ]:
# New postings per month
hist["month_posted"] = hist["first_seen"].dt.to_period("M")
monthly_new = hist.groupby("month_posted").size()

# Closings per month (last_seen for non-active jobs)
closed = hist[~hist["is_active"]].copy()
closed["month_closed"] = closed["last_seen"].dt.to_period("M")
monthly_closed = closed.groupby("month_closed").size()

fig, ax = plt.subplots(figsize=(14, 6))
months = sorted(set(monthly_new.index) | set(monthly_closed.index))
x = range(len(months))
new_vals = [monthly_new.get(m, 0) for m in months]
closed_vals = [monthly_closed.get(m, 0) for m in months]

ax.bar(x, new_vals, label="New Postings", color="#4a7cc9", alpha=0.8)
ax.bar(x, [-v for v in closed_vals], label="Closed / Filled", color="#e07b39", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels([str(m) for m in months], rotation=45, ha="right")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Monthly Hiring Activity: New Postings vs Closings")
ax.set_ylabel("Number of Roles")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Total postings ever seen: {len(hist):,}")
print(f"Peak month for new postings: {monthly_new.idxmax()} ({monthly_new.max()} roles)")

In [ ]:
# Cumulative open roles over time (running count of active postings)
# For each day, count how many jobs had first_seen <= day <= last_seen
date_range = pd.date_range(hist["first_seen"].min(), hist["last_seen"].max(), freq="W")
active_over_time = []
for d in date_range:
    n = ((hist["first_seen"] <= d) & (hist["last_seen"] >= d)).sum()
    active_over_time.append({"date": d, "active_roles": n})
df_active = pd.DataFrame(active_over_time)

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(df_active["date"], df_active["active_roles"], alpha=0.3, color="#4a7cc9")
ax.plot(df_active["date"], df_active["active_roles"], color="#4a7cc9", linewidth=2)
ax.set_title("Estimated Active Open Roles Over Time")
ax.set_ylabel("Number of Active Postings")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

print(f"Peak open roles: {df_active['active_roles'].max()} on {df_active.loc[df_active['active_roles'].idxmax(), 'date']:%Y-%m-%d}")

## 11. Department Hiring Trends

Which departments have been growing fastest? Stacked area chart of new postings by department over time.

In [ ]:
# Monthly new postings by department (all departments)
dept_monthly = (
    hist[hist["department"] != "Other"]
    .groupby(["month_posted", "department"])
    .size()
    .unstack(fill_value=0)
)

# Sort columns by total volume so largest departments are at the bottom of the stack
col_order = dept_monthly.sum().sort_values(ascending=False).index.tolist()
dept_monthly = dept_monthly[col_order]

n_depts = len(col_order)
fig, ax = plt.subplots(figsize=(18, 8))
dept_monthly.index = dept_monthly.index.to_timestamp()
cmap = plt.cm.get_cmap("tab20", n_depts)
dept_monthly.plot.area(ax=ax, alpha=0.7, linewidth=0.5,
                       color=[cmap(j) for j in range(n_depts)])
ax.set_title(f"New Postings by Department (Monthly) — {n_depts} departments")
ax.set_ylabel("New Postings")
ax.set_xlabel("")
ax.legend(title="Department", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7)
plt.tight_layout()
plt.show()

## 12. Salary Trends Over Time

Track median salary midpoint by quarter to see how compensation has evolved.

In [ ]:
# Quarterly salary stats (using first_seen as the posting date)
hist_salary["quarter"] = hist_salary["first_seen"].dt.to_period("Q")

qtr_stats = hist_salary.groupby("quarter")["mid_usd"].agg(["median", "mean", "count", "std"])
qtr_stats.index = qtr_stats.index.to_timestamp()

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# Panel 1: Median + mean salary trend
axes[0].plot(qtr_stats.index, qtr_stats["median"], "o-", label="Median", color="#4a7cc9", linewidth=2)
axes[0].plot(qtr_stats.index, qtr_stats["mean"], "s--", label="Mean", color="#e07b39", linewidth=2)
axes[0].fill_between(
    qtr_stats.index,
    qtr_stats["median"] - qtr_stats["std"],
    qtr_stats["median"] + qtr_stats["std"],
    alpha=0.15, color="#4a7cc9", label="Median ± 1 std",
)
axes[0].set_title("Salary Trends Over Time (USD, Quarterly)")
axes[0].set_ylabel("Salary Midpoint (USD)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
axes[0].legend()

# Panel 2: Number of roles with salary data per quarter
axes[1].bar(qtr_stats.index, qtr_stats["count"], width=60, color="#4a7cc9", alpha=0.7)
axes[1].set_title("Roles with Salary Data per Quarter")
axes[1].set_ylabel("Number of Roles")
axes[1].set_xlabel("")

plt.tight_layout()
plt.show()

# Print quarter-over-quarter changes
print("QUARTERLY SALARY TRENDS (USD)")
print("=" * 70)
prev_med = None
for idx, row in qtr_stats.iterrows():
    change = ""
    if prev_med is not None:
        delta = row["median"] - prev_med
        pct = delta / prev_med * 100
        change = f"  ({pct:+.1f}%)"
    print(f"  {idx:%Y Q{idx.quarter}}:  median ${row['median']:>10,.0f}  mean ${row['mean']:>10,.0f}  n={int(row['count']):>4d}{change}")
    prev_med = row["median"]

In [ ]:
# Salary trends by department (all departments with enough data)
# Require at least 2 quarters with 3+ roles for a meaningful trend
dept_qtr_counts = hist_salary.groupby(["quarter", "department"]).size().unstack(fill_value=0)
depts_with_data = [d for d in dept_qtr_counts.columns
                   if (dept_qtr_counts[d] >= 3).sum() >= 2 and d != "Other"]

dept_qtr = (
    hist_salary[hist_salary["department"].isin(depts_with_data)]
    .groupby(["quarter", "department"])["mid_usd"]
    .median()
    .unstack()
)
dept_qtr.index = dept_qtr.index.to_timestamp()

n_depts = len(depts_with_data)
fig, ax = plt.subplots(figsize=(18, 8))
cmap = plt.cm.get_cmap("tab20", n_depts)
for j, col in enumerate(dept_qtr.columns):
    ax.plot(dept_qtr.index, dept_qtr[col], "o-", label=col,
            linewidth=2, markersize=4, color=cmap(j))
ax.set_title(f"Median Salary by Department Over Time (USD, Quarterly) — {n_depts} departments")
ax.set_ylabel("Median Salary Midpoint (USD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(title="Department", fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

## 13. Time-to-Fill Analysis

How long do roles stay open? Estimated from `first_seen` to `last_seen` for closed roles.

In [ ]:
# Only consider closed roles (active roles haven't been filled yet)
closed = hist[~hist["is_active"]].copy()
closed["days_open"] = (closed["last_seen"] - closed["first_seen"]).dt.days

# Filter out roles open for 0 days (seen only in a single snapshot — imprecise)
closed_valid = closed[closed["days_open"] > 0]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Distribution of time-to-fill
sns.histplot(closed_valid["days_open"], bins=40, kde=True, ax=axes[0], color="#4a7cc9")
axes[0].axvline(closed_valid["days_open"].median(), color="red", linestyle="--", label=f"Median: {closed_valid['days_open'].median():.0f} days")
axes[0].set_title("Distribution of Time-to-Fill (Closed Roles)")
axes[0].set_xlabel("Days Open")
axes[0].set_ylabel("Number of Roles")
axes[0].legend()

# Time-to-fill by department
dept_ttf = (
    closed_valid.groupby("department")["days_open"]
    .median()
    .sort_values(ascending=False)
)
dept_ttf.plot.barh(ax=axes[1], color="#e07b39")
axes[1].set_title("Median Time-to-Fill by Department (Days)")
axes[1].set_xlabel("Median Days Open")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

print("TIME-TO-FILL STATISTICS (closed roles only)")
print("=" * 55)
print(f"  Roles analyzed: {len(closed_valid):,}")
print(f"  Median:  {closed_valid['days_open'].median():.0f} days")
print(f"  Mean:    {closed_valid['days_open'].mean():.0f} days")
print(f"  25th %%:  {closed_valid['days_open'].quantile(0.25):.0f} days")
print(f"  75th %%:  {closed_valid['days_open'].quantile(0.75):.0f} days")

## 14. Active vs Closed Role Comparison

Do closed roles differ in salary or department mix from currently active ones?

In [ ]:
hist_salary["status"] = hist_salary["is_active"].map({True: "Active", False: "Closed"})

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Salary comparison
sns.boxplot(data=hist_salary, x="status", y="mid_usd", ax=axes[0], palette="Set2")
axes[0].set_title("Salary: Active vs Closed Roles")
axes[0].set_ylabel("Salary Midpoint (USD)")
axes[0].set_xlabel("")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))

# Department mix
dept_status = hist.groupby(["department", "is_active"]).size().unstack(fill_value=0)
dept_status.columns = ["Closed", "Active"]
# Normalize to percentages
dept_pct = dept_status.div(dept_status.sum(axis=0), axis=1) * 100
dept_pct = dept_pct.sort_values("Active", ascending=True)
dept_pct.plot.barh(ax=axes[1], width=0.8)
axes[1].set_title("Department Mix: Active vs Closed (%)")
axes[1].set_xlabel("% of Roles")
axes[1].set_ylabel("")
axes[1].legend(title="Status")

# Seniority mix
sen_status = hist.groupby(["seniority", "is_active"]).size().unstack(fill_value=0)
sen_status.columns = ["Closed", "Active"]
sen_order_existing = [s for s in seniority_order if s in sen_status.index]
sen_status = sen_status.reindex(sen_order_existing)
sen_status.plot.barh(ax=axes[2], width=0.8)
axes[2].set_title("Seniority Mix: Active vs Closed")
axes[2].set_xlabel("Number of Roles")
axes[2].set_ylabel("")
axes[2].legend(title="Status")

plt.tight_layout()
plt.show()

# Print salary comparison
print("SALARY COMPARISON: ACTIVE vs CLOSED")
print("=" * 55)
for status in ["Active", "Closed"]:
    sub = hist_salary[hist_salary["status"] == status]
    print(f"  {status:8s}: n={len(sub):>4d}  median=${sub['mid_usd'].median():>10,.0f}  mean=${sub['mid_usd'].mean():>10,.0f}")

## 15. Recurring Roles: Salary Changes for Re-posted Titles

Find job titles that have been posted more than once and track how their salary ranges evolved.

In [ ]:
# Normalize titles (strip whitespace, lowercase) to find recurring roles
hist_salary["title_norm"] = hist_salary["title"].str.strip().str.lower()

# Find titles posted multiple times with salary data
title_counts = hist_salary.groupby("title_norm").agg(
    postings=("job_id", "nunique"),
    min_date=("first_seen", "min"),
    max_date=("first_seen", "max"),
    earliest_mid=("mid_usd", "first"),
    latest_mid=("mid_usd", "last"),
).query("postings >= 2")

title_counts["salary_change_pct"] = (
    (title_counts["latest_mid"] - title_counts["earliest_mid"])
    / title_counts["earliest_mid"] * 100
)
title_counts = title_counts.sort_values("postings", ascending=False)

print(f"Titles posted 2+ times (with salary): {len(title_counts)}")
print()
print("TOP RECURRING ROLES BY POSTING COUNT")
print("=" * 95)
for title_norm, row in title_counts.head(15).iterrows():
    # Get the original-cased title
    orig = hist_salary.loc[hist_salary["title_norm"] == title_norm, "title"].iloc[0]
    change_str = f"{row['salary_change_pct']:+.1f}%" if not pd.isna(row["salary_change_pct"]) else "N/A"
    print(f"  {int(row['postings'])}x  {orig[:55]:55s}  ${row['earliest_mid']:>9,.0f} -> ${row['latest_mid']:>9,.0f}  ({change_str})")

In [ ]:
# Visualize salary evolution for the most-reposted roles
top_recurring = title_counts.head(8).index.tolist()
recurring_data = hist_salary[hist_salary["title_norm"].isin(top_recurring)].copy()

fig, ax = plt.subplots(figsize=(14, 7))
for title_norm in top_recurring:
    sub = recurring_data[recurring_data["title_norm"] == title_norm].sort_values("first_seen")
    orig = sub["title"].iloc[0]
    # Truncate label
    label = orig[:40] + ("..." if len(orig) > 40 else "")
    ax.plot(sub["first_seen"], sub["mid_usd"], "o-", label=label, markersize=8, linewidth=2)

ax.set_title("Salary Evolution for Most-Reposted Roles")
ax.set_ylabel("Salary Midpoint (USD)")
ax.set_xlabel("First Seen Date")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

## 16. Historical Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle(f"{COMPANY} Historical Hiring Dashboard", fontsize=18, fontweight="bold", y=0.99)

# --- KPI banner ---
ax_kpi = fig.add_subplot(3, 2, (1, 2))
ax_kpi.axis("off")
closed_roles = hist[~hist["is_active"]]
kpis = {
    "Total Jobs Seen": f"{len(hist):,}",
    "Currently Active": f"{hist['is_active'].sum():,}",
    "Closed / Filled": f"{len(closed_roles):,}",
    "With Salary Data": f"{len(hist_salary):,}",
    "Date Range": f"{hist['first_seen'].min():%b %Y} – {hist['last_seen'].max():%b %Y}",
}
kpi_text = "    ".join(f"{k}: {v}" for k, v in kpis.items())
ax_kpi.text(0.5, 0.5, kpi_text, ha="center", va="center", fontsize=13,
            bbox=dict(boxstyle="round,pad=0.8", facecolor="#f0f4ff", edgecolor="#4a7cc9"))

# --- Active roles over time ---
ax1 = fig.add_subplot(3, 2, 3)
ax1.fill_between(df_active["date"], df_active["active_roles"], alpha=0.3, color="#4a7cc9")
ax1.plot(df_active["date"], df_active["active_roles"], color="#4a7cc9", linewidth=1.5)
ax1.set_title("Active Open Roles Over Time", fontsize=11)
ax1.set_ylabel("Roles")

# --- Salary trend ---
ax2 = fig.add_subplot(3, 2, 4)
ax2.plot(qtr_stats.index, qtr_stats["median"], "o-", color="#4a7cc9", linewidth=2, label="Median")
ax2.plot(qtr_stats.index, qtr_stats["mean"], "s--", color="#e07b39", linewidth=2, label="Mean")
ax2.set_title("Quarterly Salary Trend (USD)", fontsize=11)
ax2.set_ylabel("Salary Midpoint")
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1000:.0f}K"))
ax2.legend(fontsize=9)

# --- Department volume (all-time) ---
ax3 = fig.add_subplot(3, 2, 5)
dept_all = hist["department"].value_counts().head(12)
dept_all.sort_values().plot.barh(ax=ax3, color="#4a7cc9")
ax3.set_title("Total Postings by Department (Top 12)", fontsize=11)
ax3.set_xlabel("Total Postings")
ax3.set_ylabel("")
ax3.tick_params(axis="y", labelsize=8)

# --- Active vs closed pie ---
ax4 = fig.add_subplot(3, 2, 6)
status_counts = hist["is_active"].value_counts()
ax4.pie(
    [status_counts.get(False, 0), status_counts.get(True, 0)],
    labels=["Closed / Filled", "Active"],
    autopct="%1.0f%%", startangle=90,
    colors=["#e07b39", "#4a7cc9"],
)
ax4.set_title("Active vs Closed Roles", fontsize=11)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()